# Daily Challenge: Comprehensive Mobile Price Analysis
### Mobile Price Classification Dataset

In [ ]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

ZIP_URL = (
    "https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/"
    "Week%206%20-%20Applications%20for%20Data%20Analysis/W6D5%20-%20Mini%20project/"
    "Mobile%20Price%20Classification.zip"
)

try:
    r = requests.get(ZIP_URL, timeout=30)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        train_file = [f for f in z.namelist() if 'train' in f.lower() and f.endswith('.csv')][0]
        with z.open(train_file) as f:
            df = pd.read_csv(f)
    print("Loaded from URL.")
except Exception as e:
    print(f"URL failed ({e}). Falling back to local file.")
    df = pd.read_csv('train.csv')

print(f"Shape: {df.shape}")
df.head()

## 1. Data Loading and Exploration

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print(f"\nTarget variable: 'price_range'")
print(f"  Classes: {sorted(df['price_range'].unique())}  (0=low, 1=medium, 2=high, 3=very high)")
print(f"\nFeatures: {df.shape[1] - 1}")
print(f"Samples : {df.shape[0]}")

In [ ]:
print("=== Descriptive Statistics ===")
df.describe().round(2)

In [ ]:
print("Target class distribution:")
print(df['price_range'].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(6, 4))
df['price_range'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color=['steelblue', 'coral', 'seagreen', 'orchid'], edgecolor='white'
)
ax.set_title('Distribution of Price Range Classes')
ax.set_xlabel('Price Range')
ax.set_ylabel('Count')
ax.set_xticklabels(['Low (0)', 'Medium (1)', 'High (2)', 'Very High (3)'], rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Data Cleaning and Preprocessing

In [ ]:
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values found.")

print("\n=== Duplicate Rows ===")
print(f"Duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()

In [ ]:
# Binary columns are already 0/1; no further encoding needed.
# Identify binary vs continuous features
binary_cols     = [c for c in df.columns if df[c].nunique() == 2 and c != 'price_range']
continuous_cols = [c for c in df.columns if c not in binary_cols and c != 'price_range']

print(f"Binary features     : {binary_cols}")
print(f"Continuous features : {continuous_cols}")

## 3. Statistical Analysis with NumPy and SciPy

In [ ]:
# Central tendency, variability, and distribution shape for each continuous feature
rows = []
for col in continuous_cols:
    vals = df[col].dropna().values
    mode_val = stats.mode(vals, keepdims=True).mode[0]
    rows.append({
        'feature'  : col,
        'mean'     : np.mean(vals),
        'median'   : np.median(vals),
        'mode'     : mode_val,
        'range'    : np.ptp(vals),
        'variance' : np.var(vals, ddof=1),
        'std'      : np.std(vals, ddof=1),
        'skewness' : stats.skew(vals),
        'kurtosis' : stats.kurtosis(vals)
    })

stat_df = pd.DataFrame(rows).set_index('feature').round(4)
print("=== Statistical Profile of Continuous Features ===")
stat_df

In [ ]:
# Hypothesis testing: one-way ANOVA — does RAM differ across price range classes?
groups = [df[df['price_range'] == cls]['ram'].values for cls in sorted(df['price_range'].unique())]
f_stat, p_anova = stats.f_oneway(*groups)

print("One-way ANOVA — RAM across price range classes:")
print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value    : {p_anova:.2e}")
if p_anova < 0.05:
    print("  Conclusion: RAM differs significantly across price range classes (p < 0.05).")
else:
    print("  Conclusion: No significant difference in RAM across classes.")

In [ ]:
# Feature-target correlation using SciPy's Spearman (handles non-linear monotonic relationships)
corr_rows = []
target = df['price_range'].values
for col in continuous_cols:
    r, p = stats.spearmanr(df[col].values, target)
    corr_rows.append({'feature': col, 'spearman_r': round(r, 4), 'p_value': round(p, 6)})

corr_df = pd.DataFrame(corr_rows).set_index('feature').sort_values('spearman_r', key=abs, ascending=False)
print("Spearman correlation with price_range:")
print(corr_df)

## 4. Data Visualization with Matplotlib

In [ ]:
# Histograms for key continuous features
key_features = ['ram', 'battery_power', 'px_height', 'px_width', 'int_memory', 'mobile_wt']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(key_features):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='none', alpha=0.8)
    axes[i].axvline(df[col].mean(),   color='crimson', linewidth=1.5, linestyle='--', label='Mean')
    axes[i].axvline(df[col].median(), color='orange',  linewidth=1.5, linestyle=':',  label='Median')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Distribution of Key Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: RAM and battery_power by price range
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = ['Low', 'Medium', 'High', 'Very High']

for ax, col in zip(axes, ['ram', 'battery_power']):
    data = [df[df['price_range'] == cls][col].values for cls in sorted(df['price_range'].unique())]
    ax.boxplot(data, labels=labels, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='crimson', linewidth=2))
    ax.set_title(f'{col} by Price Range')
    ax.set_xlabel('Price Range')
    ax.set_ylabel(col)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Feature Distribution by Price Range', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: RAM vs battery_power colored by price_range
colors = {0: 'steelblue', 1: 'coral', 2: 'seagreen', 3: 'orchid'}
price_labels = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Very High'}

fig, ax = plt.subplots(figsize=(10, 6))
for cls in sorted(df['price_range'].unique()):
    subset = df[df['price_range'] == cls]
    ax.scatter(subset['ram'], subset['battery_power'],
               color=colors[cls], label=price_labels[cls], alpha=0.4, s=10)

ax.set_title('RAM vs Battery Power by Price Range')
ax.set_xlabel('RAM (MB)')
ax.set_ylabel('Battery Power (mAh)')
ax.legend(title='Price Range')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr_matrix = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, linewidths=0.5,
    annot_kws={'size': 7}, ax=ax
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Insight Synthesis and Conclusion

In [ ]:
# Top features correlated with price_range
top_features = corr_df.head(5)

fig, ax = plt.subplots(figsize=(8, 4))
colors_bar = ['seagreen' if v > 0 else 'crimson' for v in top_features['spearman_r']]
ax.barh(top_features.index[::-1], top_features['spearman_r'][::-1],
        color=colors_bar[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 5 Features by Spearman Correlation with Price Range')
ax.set_xlabel('Spearman r')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 55)
print("       CONCLUSIONS — MOBILE PRICE ANALYSIS")
print("=" * 55)
print()
print("1. KEY DETERMINANT:")
print(f"   RAM is by far the strongest predictor of price range")
print(f"   (Spearman r = {corr_df.loc['ram', 'spearman_r']:.3f}, p < 0.001).")
print(f"   Higher RAM is consistently associated with higher price.")
print()
print("2. BATTERY POWER:")
print(f"   Positively correlated with price (r = {corr_df.loc['battery_power', 'spearman_r']:.3f}).")
print(f"   Higher-end phones tend to have larger batteries.")
print()
print("3. PIXEL DIMENSIONS:")
print("   Screen resolution (px_height, px_width) shows a moderate")
print("   positive correlation with price range.")
print()
print("4. WEIGHT:")
print(f"   Mobile weight shows a slight negative or near-zero correlation,")
print(f"   suggesting lighter phones are not necessarily more expensive.")
print()
print("5. BALANCED DATASET:")
print("   The dataset is perfectly balanced (500 samples per class),")
print("   making it well-suited for classification without resampling.")
print()
print("6. ANOVA RESULT:")
print(f"   F = {f_stat:.2f}, p = {p_anova:.2e} → RAM distribution differs")
print("   highly significantly across all four price classes.")